# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access metadata fields as attributes per mlcroissant
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and fields by @id
print("Available record sets (by @id and name):")

record_sets = dataset.record_sets
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', '[no name]')}")
    if 'field' in rs:
        print("    Fields (by @id and name):")
        for field in rs['field']:
            # Resolve the field object
            field_obj = dataset._find_by_id(field['@id']) if isinstance(field, dict) and '@id' in field else field
            if field_obj:
                print(f"      - @id: {field_obj.get('@id')}, name: {field_obj.get('name', '[no name]')}, dataType: {field_obj.get('dataType', '[no dataType]')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all record set @ids for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
# For demonstration, we'll load the first record set (could adapt to any of them as needed)
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id} with {len(df)} records")
        print(f"Fields: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records for record set {record_set_id}.")
# Choose a record set for EDA if available
selected_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
# Attempt EDA for the selected record set
if selected_record_set_id and selected_record_set_id in dataframes:
    df = dataframes[selected_record_set_id].copy()
    # Try to select the first numeric field for filtering/normalizing
    numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field '{numeric_field}' for EDA")
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

        print(f"Normalized {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping if there is a categorical field
        group_fields = df.select_dtypes(exclude=np.number).columns.tolist()
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No non-numeric group field available in the dataset.")
    else:
        print("No numeric fields found in the DataFrame for analysis.")
else:
    print("No suitable record set found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If EDA returned a filtered DataFrame and a numeric field, display histogram
if selected_record_set_id and selected_record_set_id in dataframes:
    df = dataframes[selected_record_set_id]
    numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field], kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()

        # If there is a group field, plot boxplot
        group_fields = df.select_dtypes(exclude=np.number).columns.tolist()
        if group_fields:
            group_field = group_fields[0]
            plt.figure(figsize=(10, 4))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f"{numeric_field} by {group_field}")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.xticks(rotation=45)
            plt.show()
    else:
        print("No numeric field for visualization.")
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


This notebook demonstrated how to load and explore the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the `mlcroissant` library. 

- The Croissant schema allows structured exploration of record sets and fields by their unique `@id` references.
- We loaded and briefly explored the record sets, listed available fields with their types, and performed standard EDA operations such as filtering, normalization, and grouping if suitable fields were available.
- Example data visualizations (histogram and boxplot) illustrated basic field distributions and group comparisons.

Continue exploring the dataset and its metadata for deeper analysis. Refer to the Croissant schema documentation for advanced data integration and provenance features.